# EDA_PIPELINE

In [9]:
#IMPORTACIONES  

from src.spark import get_spark
from src.io import read_parquet, write_parquet, to_csv_via_pandas
from src.profiling import (
    show_head_and_schema,
    describe_df,
    report_missing,
    count_row_duplicates,
    count_missing_or_nan,
    count_zeros,
    count_distinct_row_duplicates,
)
from src.outliers import numeric_columns, mad_outlier_report, iqr_outlier_report
from src.cleaning import drop_columns, dropna_subset
import src.config as cfg


In [10]:
import sys
from pathlib import Path

ROOT = Path("/home/jovyan/work")   
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [11]:
# Spark + lectura
spark = get_spark(cfg.SPARK_APP_NAME)

behavioural_df = read_parquet(spark, cfg.BEHAVIOURAL_PATH_RAW)
clientes_df = read_parquet(spark, cfg.CLIENTS_PATH_RAW)


## ANALISIS BEHAVIOURAL

Descripción y visualización de cada variable y análisis de existencia de duplicados y missing values.

In [12]:
# EDA INICIAL 

show_head_and_schema(behavioural_df, "BEHAVIOURAL", n=5, truncate=False)
describe_df(behavioural_df, "BEHAVIOURAL")
report_missing(behavioural_df, "BEHAVIOURAL")
count_row_duplicates(behavioural_df, "BEHAVIOURAL")

beh_cols = numeric_columns(behavioural_df)


BEHAVIOURAL - head(5)
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821016961u00XXX|ES182147947X|2020-08-22|0.0                 |2700.0           |0.0                     |0.0                 |0.0                     |0.0                      

Análisis de outliers.

In [13]:
# Outliers (MAD)
mad_res = mad_outlier_report(behavioural_df, cols=beh_cols, threshold=3.5, ignore_zeros=True)
# Outliers (IQR)
iqr_res = iqr_outlier_report(behavioural_df, cols=beh_cols, factor=1.5)

In [ ]:
# Imprime top columnas por % outliers
top_mad = sorted(mad_res.items(), key=lambda kv: (kv[1].get("outlier_pct") or 0), reverse=True)[:10]
print("\nTop 10 columnas por % outliers (MAD):")
for c, r in top_mad:
    print(c, r.get("outlier_pct"), "%")


Top 10 columnas por % outliers (MAD):
NUMBER_DRAWINGS 14.238816882444706 %
CREDIT_CARD_DRAWINGS 13.135128548799804 %
CREDIT_CARD_LIMIT 13.088779840561893 %
CREDIT_CARD_DRAWINGS_ATM 13.022402590747914 %
CREDIT_CARD_DRAWINGS_OTHER 13.000690607734807 %
CREDIT_CARD_DRAWINGS_POS 12.102118600418912 %
CREDIT_CARD_PAYMENT 6.823045091371177 %
CREDICT_CARD_BALANCE 4.882447831973931 %
NUMBER_INSTALMENTS 0.4919975544283846 %
NUMBER_DRAWINGS_ATM 0.0 %


Realizamos la limpieza eliminando columnas como 'CURRENCY' que nos indica que la moneda es el 'euro' y ese dato es común en ambos datasets para todos los clientes. Además eliminamos otras columnas que no son relevantes para el estudio o que por la presencia alta de missing values no nos permiten analizar correctamente al clientes. 

Respecto a los outliers los hemos dejado tal y como están debido al desbalance del dataset y a que toda información es relevante sobre los clientes.

In [15]:
# Limpieza BEHAVIOURAL
behavioural_df_clean = drop_columns(behavioural_df, cfg.BEH_DROP_COLS)

## ANALISIS CLIENTS

Descripción y visualización de cada variable y análisis de existencia de duplicados y missing values.

In [17]:
show_head_and_schema(clientes_df, "CLIENTS", n=5, truncate=False)
describe_df(clientes_df, "CLIENTS")
report_missing(clientes_df, "CLIENTS")
count_distinct_row_duplicates(clientes_df, "CLIENTS")


CLIENTS - head(5)
+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_SCORE|AGE_I

0

Dado que hay missing values se obtine el número y el porcentaje por columna de estos.

In [18]:
count_missing_or_nan(clientes_df, "CLIENTS")
count_zeros(clientes_df, "CLIENTS")


CLIENTS - missing (alt)
CLIENT_ID: 0 missing (0.00%)
NON_COMPLIANT_CONTRACT: 0 missing (0.00%)
NAME_PRODUCT_TYPE: 0 missing (0.00%)
GENDER: 0 missing (0.00%)
TOTAL_INCOME: 0 missing (0.00%)
AMOUNT_PRODUCT: 0 missing (0.00%)
INSTALLMENT: 7 missing (0.00%)
EDUCATION: 39640 missing (24.32%)
MARITAL_STATUS: 2 missing (0.00%)
HOME_SITUATION: 0 missing (0.00%)
REGION_SCORE: 0 missing (0.00%)
AGE_IN_YEARS: 0 missing (0.00%)
JOB_SENIORITY: 29174 missing (17.90%)
HOME_SENIORITY: 0 missing (0.00%)
LAST_UPDATE: 0 missing (0.00%)
OWN_INSURANCE_CAR: 0 missing (0.00%)
CAR_AGE: 107550 missing (65.99%)
FAMILY_SIZE: 2 missing (0.00%)
REACTIVE_SCORING: 91901 missing (56.39%)
PROACTIVE_SCORING: 337 missing (0.21%)
BEHAVIORAL_SCORING: 32246 missing (19.79%)
DAYS_LAST_INFO_CHANGE: 1 missing (0.00%)
NUMBER_OF_PRODUCTS: 21903 missing (13.44%)
OCCUPATION: 0 missing (0.00%)
DIGITAL_CLIENT: 0 missing (0.00%)
HOME_OWNER: 0 missing (0.00%)
EMPLOYER_ORGANIZATION_TYPE: 29464 missing (18.08%)
CURRENCY: 0 missing (0

{'CLIENT_ID': 0,
 'NON_COMPLIANT_CONTRACT': 149741,
 'NAME_PRODUCT_TYPE': 0,
 'GENDER': 0,
 'TOTAL_INCOME': 0,
 'AMOUNT_PRODUCT': 0,
 'INSTALLMENT': 0,
 'EDUCATION': 0,
 'MARITAL_STATUS': 0,
 'HOME_SITUATION': 0,
 'REGION_SCORE': 0,
 'AGE_IN_YEARS': 0,
 'JOB_SENIORITY': 0,
 'HOME_SENIORITY': 43,
 'LAST_UPDATE': 7,
 'OWN_INSURANCE_CAR': 0,
 'CAR_AGE': 1136,
 'FAMILY_SIZE': 0,
 'REACTIVE_SCORING': 0,
 'PROACTIVE_SCORING': 0,
 'BEHAVIORAL_SCORING': 0,
 'DAYS_LAST_INFO_CHANGE': 19947,
 'NUMBER_OF_PRODUCTS': 38081,
 'OCCUPATION': 0,
 'DIGITAL_CLIENT': 153832,
 'HOME_OWNER': 0,
 'EMPLOYER_ORGANIZATION_TYPE': 0,
 'CURRENCY': 0,
 'NUM_PREVIOUS_LOAN_APP': 0,
 'LOAN_ANNUITY_PAYMENT_MAX': 200,
 'LOAN_ANNUITY_PAYMENT_MIN': 67707,
 'LOAN_ANNUITY_PAYMENT_SUM': 200,
 'LOAN_APPLICATION_AMOUNT_MAX': 529,
 'LOAN_APPLICATION_AMOUNT_MIN': 72400,
 'LOAN_APPLICATION_AMOUNT_SUM': 529,
 'LOAN_CREDIT_GRANTED_MAX': 112,
 'LOAN_CREDIT_GRANTED_MIN': 60036,
 'LOAN_CREDIT_GRANTED_SUM': 112,
 'LOAN_VARIABLE_RATE_MAX

Realizamos la limpieza eliminando columnas como 'CURRENCY' que nos indica que la moneda es el 'euro' y ese dato es común en ambos datasets para todos los clientes. Además eliminamos otras columnas que no son relevantes para el estudio o que por la presencia alta de missing values no nos permiten analizar correctamente al clientes. 

Respecto a los outliers los hemos dejado tal y como están debido al desbalance del dataset y a que toda información es relevante sobre los clientes.

In [19]:
# Limpieza CLIENTS
clientes_df_clean = dropna_subset(clientes_df, cfg.CLI_DROPNA_SUBSET)
clientes_df_clean = drop_columns(clientes_df_clean, cfg.CLI_DROP_COLS)

Guardado y exportación de datasets.

In [20]:
# Guardado de parquets limpios
write_parquet(behavioural_df_clean, cfg.BEHAVIOURAL_PATH_CLEAN, mode="overwrite")
write_parquet(clientes_df_clean, cfg.CLIENTS_PATH_CLEAN, mode="overwrite")

In [21]:
# Exportamos CSV vía pandas
df_cli = read_parquet(spark, cfg.CLIENTS_PATH_CLEAN)
df_beh = read_parquet(spark, cfg.BEHAVIOURAL_PATH_CLEAN)

to_csv_via_pandas(df_cli, cfg.CLIENTS_CSV_CLEAN, index=False)
to_csv_via_pandas(df_beh, cfg.BEHAVIOURAL_CSV_CLEAN, index=False)  